In [3]:
from functools import partial
from notebooks._utils import report_series_ensemble_accuracy_by_nparas
from notebooks._utils import calculate_parallel_ensemble_accuracy
from notebooks._utils import calculate_baseline_accuracy, get_layers

ds_name = "myriadlama"
dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"

In [4]:


num_fewshots = 5
# for model_name in ["llama3.2_1b", "llama3.2_1b_it", "qwen2.5_14b", "qwen2.5_14b_it"]:
# , "qwen2.5_3b", "qwen2.5_3b_it", "qwen2.5_7b", "qwen2.5_7b_it", "qwen2.5_14b", "qwen2.5_14b_it"
# for model_name in ["llama3.2_1b", "llama3.2_1b_it", "llama3.2_3b", "llama3.2_3b_it", "llama3.1_8b", "llama3.1_8b_it"]:
for model_name in ["qwen3_30b", "qwen3_4b", "pythia_2.8b", "llama3.2_3b", "qwen2.5_3b"]:
    print(f"\n=================== Model: {model_name} ===================")
    dump_file_prefix = f"{dataset_root}/{ds_name}/{model_name}/myriadlama."
    report_accuracy = partial(
        report_series_ensemble_accuracy_by_nparas, 
        dump_file_prefix=dump_file_prefix,
        single_para_qapair=True,
        explicit_prompts=False,
        repeat_paras=False, 
        num_fewshots=num_fewshots)
    
    print("---- Calculating baseline ----")
    calculate_baseline_accuracy(dataset_root, model_name, num_fewshots)

    print("---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----")
    report_accuracy(modifyattn=False, modifyrope=False, scale_score=False)

    print("---- ✅ attention mask / ✅ rope modifications / ✅ scale score ----")
    report_accuracy(modifyattn=True, modifyrope=True, scale_score=True)

    print("---- Logits Ensemble  ----")
    calculate_parallel_ensemble_accuracy(dump_file_prefix, repeat_paras=False,
        num_paraphrases=5, num_fewshots=num_fewshots,
        logits_ensemble_method="avg")
    
    print("---- Logits Ensemble + Layer Average ----")
    calculate_parallel_ensemble_accuracy(dump_file_prefix, repeat_paras=False,
        num_paraphrases=5, num_fewshots=num_fewshots,
        logits_ensemble_method="avg",
        ensemble_method="layer_output_avg",
        multilayer=True, ensemble_alpha=1.0, 
        ensemble_layer=get_layers(model_name),
        token_mode="last", use_generation=True)
    


=================== Model: qwen3_30b ===================
---- Calculating baseline ----
Acc: 0.5341 ==> 🏷️ baseline
---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----
File ./singleparaqapair.5samples.5paras.feather does not exist!
---- ✅ attention mask / ✅ rope modifications / ✅ scale score ----
File ./modifyattn.modifyrope.scalescore.singleparaqapair.5samples.5paras.feather does not exist!
---- Logits Ensemble  ----
File ./logits.avg.5samples.5paras.feather does not exist!
---- Logits Ensemble + Layer Average ----
File ./logits.avg.avglayer.layer36.alpha100.token-last.multilayer.5samples.5paras.feather does not exist!

=================== Model: qwen3_4b ===================
---- Calculating baseline ----
Acc: 0.3915 ==> 🏷️ baseline
---- ❌ attention mask / ❌ rope modifications / ❌ scale score ----
File ./singleparaqapair.5samples.5paras.feather does not exist!
---- ✅ attention mask / ✅ rope modifications / ✅ scale score ----
File ./modifyattn.modifyrope.scalescore.singl

In [ ]:
_calculate_accuracy(max_df, "Logits-based Ensemble (max)")
_calculate_accuracy(avg_df, "Logits-based Ensemble (avg)")